**420-A58-SF - Algorithmes d'apprentissage non supervisé - Hiver 2026 - Spécialisation technique en Intelligence Artificielle**<br/>
MIT License - Copyright (c) 2026 Mikaël Swawola

# Examen #2 - Locality-Sensitive Hashing (LSH)

| | |
|---|---|
| **Durée** | 3 heures 30 minutes |
| **Travail** | Individuel |
| **Documents** | Autorisés |
| **Remise** | Remettre ce notebook complété sur Lea |

## Engagement
En soumettant ce travail, je certifie sur l’honneur avoir respecté l’ensemble des règles d’intégrité académique et ne pas avoir eu recours à des logiciels d’intelligence artificielle générative ni à l’aide de tiers pour le réaliser. Je comprends que toute violation de ces règles peut entraîner des sanctions académiques.

Inscrirez votre nom complet : _____________________________

Date : _____________________________

Tout refus de signer cet engagement entraînera une note nulle à cette évaluation sommative.

## Contexte

La recherche de voisins les plus proches (*nearest neighbors*) est un problème fondamental en intelligence artificielle. Lorsque le nombre de documents et la dimension des données sont très grands, une recherche par force brute devient trop coûteuse. L'algorithme **Locality-Sensitive Hashing (LSH)** permet d'effectuer une recherche approximative de voisins de manière efficace en partitionnant les données dans des **bins** à l'aide de projections binaires aléatoires.

Dans cette évaluation, vous implémenterez et analyserez l'algorithme LSH appliqué au jeu de données **people_wiki**, constitué de pages Wikipedia de personnalités représentées sous forme de vecteurs TF-IDF.

## Données

Le dossier `data/people/` contient les fichiers suivants :
- **people_wiki.csv** : jeu de données constitué des pages Wikipedia de personnalités
- **people_wiki_map_index_to_word.json** : mapping entre les mots et les indices du vocabulaire
- **people_wiki_tf_idf.npz** : vecteurs TF-IDF pour chaque document (matrice creuse)

## Barème

| Section | Question | Points |
|---------|----------|--------|
| **Partie A — Travail préliminaire et projection binaire (40 points)** | | |
| 1 - Exploration des données | Q1.1 - Chargement des données | 3 |
| | Q1.2 - Dimensions et aperçu | 3 |
| | Q1.3 - Interprétation de la matrice TF-IDF | 4 |
| 2 - Projection binaire aléatoire | Q2.1 - Génération des vecteurs aléatoires | 3 |
| | Q2.2 - Calcul du premier bit | 5 |
| | Q2.3 - Calcul de tous les bits d'un document | 5 |
| | Q2.4 - Rôle du signe du produit scalaire | 5 |
| 3 - De bits à bins | Q3.1 - Conversion en indice décimal | 5 |
| | Q3.2 - Nombre maximal de bins | 3 |
| | Q3.3 - Intuition sur les bins et la similarité | 4 |
| **Partie B — Entraînement et inspection du modèle (30 points)** | | |
| 4 - Table de hachage | Q4.1 - Compléter la fonction `train_lsh` | 5 |
| | Q4.2 - Entraînement et vérification | 4 |
| | Q4.3 - Bin de Barack Obama | 4 |
| | Q4.4 - Comparaison Obama vs Biden | 5 |
| 5 - Similarité cosinus | Q5.1 - Distance cosinus dans le même bin | 5 |
| | Q5.2 - Distance cosinus Obama-Biden | 4 |
| | Q5.3 - Interprétation des résultats | 3 |
| **Partie C — Recherche de plus proches voisins (30 points)** | | |
| 6 - Bins voisins | Q6.1 - Inversion de bits | 6 |
| | Q6.2 - Mise à jour des candidats | 6 |
| | Q6.3 - Nombre de bins voisins | 6 |
| 7 - Fonction de requête | Q7.1 - Calcul des distances | 6 |
| | Q7.2 - Requête pour Barack Obama | 6 |
| **Total** | | **100** |

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from copy import copy
from itertools import combinations
from sklearn.metrics.pairwise import pairwise_distances

In [ ]:
# Fonction utilitaire fournie - Chargement de matrice creuse
def load_sparse_csr(filename):
    loader = np.load(filename)
    data = loader['data']
    indices = loader['indices']
    indptr = loader['indptr']
    shape = loader['shape']
    return csr_matrix((data, indices, indptr), shape)

# Fonction utilitaire fournie - Distance cosinus
def cosine_distance(x, y):
    xy = x.dot(y.T)
    norm_x = np.sqrt(x.dot(x.T))
    norm_y = np.sqrt(y.dot(y.T))
    return 1 - xy[0,0] / (norm_x[0,0] * norm_y[0,0])

---
# Partie A — Travail préliminaire et projection binaire aléatoire (40 points)

---
## 1 - Exploration des données (10 points)

**Q1.1 - Charger le fichier `people_wiki.csv` dans un DataFrame Pandas, la matrice TF-IDF (fichier `people_wiki_tf_idf.npz`) et le dictionnaire de mapping des mots (fichier `people_wiki_map_index_to_word.json`). Les fichiers se trouvent dans le dossier `data/people/`.** (3 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>pd.read_csv()</code>, <code>load_sparse_csr()</code> et <code>json.load()</code></p>
</details>

In [ ]:
# Compléter le code ~ 4-5 lignes de code

**Q1.2 - Afficher les 5 premières lignes du DataFrame ainsi que les dimensions de la matrice TF-IDF (nombre de documents et taille du vocabulaire).** (3 points)

In [ ]:
# Compléter le code ~ 3 lignes de code

**Q1.3 - Expliquer en vos propres mots ce que représente chaque ligne et chaque colonne de la matrice TF-IDF. Pourquoi utilise-t-on une matrice creuse (*sparse*) pour stocker ces données ?** (4 points)

Votre réponse ici.

---
## 2 - Projection binaire aléatoire (18 points)

LSH utilise des **vecteurs aléatoires** pour partitionner l'espace des documents en bins. Chaque vecteur aléatoire définit un hyperplan qui divise l'espace en deux demi-espaces. Le **signe du produit scalaire** entre un document et un vecteur aléatoire détermine de quel côté de l'hyperplan se trouve le document.

La fonction suivante génère une collection de vecteurs aléatoires à partir d'une distribution gaussienne :

In [ ]:
# Fonction fournie
def generate_random_vectors(num_vector, dim):
    return np.random.randn(dim, num_vector)

**Q2.1 - En utilisant un seed de 2020 et la fonction `generate_random_vectors`, générer 16 vecteurs aléatoires de même dimension que la taille du vocabulaire. Afficher la forme (*shape*) du résultat.** (3 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>La dimension correspond au nombre de colonnes de la matrice TF-IDF, soit <code>len(map_index_to_word)</code></p>
</details>

In [ ]:
# Compléter le code ~ 3 lignes de code

**Q2.2 - Extraire le document d'indice 0 de la matrice TF-IDF. Calculer le produit scalaire entre ce document et le premier vecteur aléatoire. Le résultat est-il positif ou négatif ? En déduire la valeur du premier bit (1 si positif ou nul, 0 si négatif).** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>corpus[0, :]</code> pour le document et <code>random_vectors[:, 0]</code> pour le premier vecteur, puis <code>.dot()</code></p>
</details>

In [ ]:
# Compléter le code ~ 3-4 lignes de code

**Q2.3 - De manière vectorisée, calculer les 16 bits du bin correspondant au document d'indice 0. Utiliser des entiers (0 ou 1) pour représenter les bits.** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Calculer <code>doc.dot(random_vectors)</code> puis vérifier <code>>= 0</code> et convertir en entier avec <code>dtype=int</code></p>
</details>

In [ ]:
# Compléter le code ~ 1-2 lignes de code

**Q2.4 - Expliquer en vos propres mots pourquoi le signe du produit scalaire entre un document et un vecteur aléatoire permet de partitionner l'espace des documents. Pourquoi des documents similaires (proches dans l'espace TF-IDF) ont-ils tendance à obtenir les mêmes bits ?** (5 points)

Votre réponse ici.

---
## 3 - De bits à bins (12 points)

Chaque document est maintenant représenté par un vecteur de 16 bits. Pour indexer facilement les bins, on convertit ce vecteur binaire en un entier décimal.

**Q3.1 - Calculer les indices de bins (en format décimal) pour les 10 premiers documents de la matrice TF-IDF.** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>D'abord calculer tous les bits : <code>index_bits = corpus.dot(random_vectors) >= 0</code><br/>
Puis utiliser les puissances de 2 : <code>powers_of_two = 1 << np.arange(15, -1, -1)</code><br/>
Enfin : <code>index_bits.dot(powers_of_two)</code></p>
</details>

In [ ]:
# Compléter le code ~ 3-4 lignes de code

**Q3.2 - Avec un encodage sur 16 bits, combien de bins distincts peut-on avoir au maximum ? Justifier.** (3 points)

Votre réponse ici.

**Q3.3 - Expliquer intuitivement pourquoi des documents similaires (ayant un contenu textuel proche) ont tendance à se retrouver dans le même bin ou dans des bins voisins.** (4 points)

Votre réponse ici.

---
# Partie B — Entraînement et inspection du modèle LSH (30 points)

---
## 4 - Construction de la table de hachage (18 points)

La table de hachage est un dictionnaire Python qui associe à chaque indice de bin la liste des identifiants (ids) des documents qui s'y trouvent.

**Q4.1 - Compléter la fonction `train_lsh` ci-dessous. Les deux parties à compléter sont indiquées par des commentaires `# ICI`.** (5 points)

La logique à implémenter dans la boucle :
1. Si le bin n'existe pas encore dans la table, créer une liste vide
2. Ajouter l'identifiant du document à la liste du bin correspondant

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>if bin_index not in table: table[bin_index] = []</code> puis <code>table[bin_index].append(data_index)</code></p>
</details>

In [ ]:
def train_lsh(data, num_vector=16, seed=2020):
    
    dim = data.shape[1]
    if seed is not None:
        np.random.seed(seed)
    
    # Génération des vecteurs aléatoires
    random_vectors = generate_random_vectors(num_vector, dim)
    
    # Calcul des bits pour tous les documents
    bin_index_bits = (data.dot(random_vectors) >= 0)
  
    # Conversion en indices décimaux
    powers_of_two = 1 << np.arange(num_vector-1, -1, -1)
    bin_indices = bin_index_bits.dot(powers_of_two)
    
    # Construction de la table de hachage
    table = {}
    
    for data_index, bin_index in enumerate(bin_indices):
        # ------------------------------------------------------------------
        # ICI: Si le bin n'existe pas dans la table, créer une liste vide
        # ------------------------------------------------------------------
        pass
        # ------------------------------------------------------------------
        # ICI: Ajouter data_index à la liste du bin
        # ------------------------------------------------------------------
        pass

    model = {'data': data,
             'bin_index_bits': bin_index_bits,
             'bin_indices': bin_indices,
             'table': table,
             'random_vectors': random_vectors,
             'num_vector': num_vector}
    
    return model

**Q4.2 - Entraîner le modèle LSH avec 16 vecteurs et un seed de 143. Afficher le nombre total de bins créés dans la table.** (4 points)

In [ ]:
# Compléter le code ~ 2 lignes de code

**Q4.3 - Quel est l'indice du bin contenant le document de Barack Obama ? L'identifiant (id) du document de Barack Obama peut être obtenu à partir du DataFrame.** (4 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>wiki[wiki['name'] == 'Barack Obama'].index.values[0]</code> pour obtenir l'id, puis <code>model['bin_indices'][id]</code></p>
</details>

In [ ]:
# Compléter le code ~ 2-3 lignes de code

**Q4.4 - Les documents de Barack Obama et Joe Biden sont-ils dans le même bin ? Comparer leur représentation binaire (les 16 bits). Sur combien de bits diffèrent-ils ?** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Comparer <code>model['bin_index_bits'][id_obama]</code> et <code>model['bin_index_bits'][id_biden]</code></p>
</details>

In [ ]:
# Compléter le code ~ 4-10 lignes de code

---
## 5 - Similarité cosinus (12 points)

Vérifions si les documents du même bin qu'Obama sont réellement les plus proches de lui.

**Q5.1 - Lister les documents présents dans le même bin que Barack Obama (excluant Obama lui-même). Pour chacun, afficher le nom et la distance cosinus avec Obama.** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>model['table'][model['bin_indices'][id_obama]]</code> pour obtenir les ids du bin, puis <code>cosine_distance()</code></p>
</details>

In [ ]:
# Compléter le code ~ 6-8 lignes de code

**Q5.2 - Calculer la distance cosinus entre Barack Obama et Joe Biden.** (4 points)

In [ ]:
# Compléter le code ~ 2-3 lignes de code

**Q5.3 - Joe Biden est-il plus proche (au sens de la distance cosinus) de Barack Obama que les documents présents dans le bin d'Obama ? Expliquer pourquoi un document très similaire peut se retrouver dans un bin différent.** (3 points)

Votre réponse ici.

---
# Partie C — Recherche de plus proches voisins (30 points)

Comme nous l'avons observé, se limiter au bin exact de la requête peut manquer des voisins proches. La solution est de **chercher aussi dans les bins voisins**, c'est-à-dire les bins dont la représentation binaire diffère de quelques bits.

---
## 6 - Recherche dans les bins voisins (18 points)

**Q6.1 et Q6.2 - Compléter la fonction `search_nearby_bins` ci-dessous. Deux parties sont à compléter (indiquées par `# ICI`) :** (12 points)

1. **Q6.1** (6 points) : Inverser le bit à la position `i` dans le vecteur `alternate_bits`
2. **Q6.2** (6 points) : Ajouter les documents du bin voisin à l'ensemble `candidate_set`

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Q6.1 : <code>alternate_bits[i] = not alternate_bits[i]</code><br/>
Q6.2 : Utiliser <code>candidate_set.update(table[nearby_bin])</code></p>
</details>

In [ ]:
def search_nearby_bins(query_bin_bits, table, search_radius=2, initial_candidates=set()):
    """
    Pour un vecteur requête donné et un modèle LSH entraîné,
    retourne tous les candidats voisins de la requête dans les bins du rayon de recherche.
    """
    num_vector = len(query_bin_bits)
    powers_of_two = 1 << np.arange(num_vector-1, -1, -1)
    
    candidate_set = copy(initial_candidates)
    
    for different_bits in combinations(range(num_vector), search_radius):
        alternate_bits = copy(query_bin_bits)
        for i in different_bits:
            # ------------------------------------------------------------------
            # ICI (Q6.1): Inverser le bit à la position i ~ 1 ligne
            # ------------------------------------------------------------------
            pass
        
        # Conversion en indice décimal
        nearby_bin = alternate_bits.dot(powers_of_two)
        
        if nearby_bin in table:
            # ------------------------------------------------------------------
            # ICI (Q6.2): Ajouter les documents du bin à candidate_set ~ 1 ligne
            # ------------------------------------------------------------------
            pass
            
    return candidate_set

**Q6.3 - Pour un encodage sur 16 bits :** (6 points)

a) Combien de bins voisins faut-il explorer pour un rayon de recherche de 1 ?

b) Combien pour un rayon de 2 ?

c) Combien pour un rayon de 3 ?

d) Quelle formule mathématique (combinaison) permet de calculer le nombre de bins à explorer pour un rayon $r$ avec $n$ bits ?

e) Que se passe-t-il si on augmente le rayon de recherche jusqu'à 16 ?

Votre réponse ici.

---
## 7 - Fonction de requête (12 points)

La fonction `query` combine la recherche dans les bins voisins avec le calcul des distances réelles pour retourner les $k$ plus proches voisins.

**Q7.1 - Compléter la ligne manquante dans la fonction `query` ci-dessous pour calculer les distances cosinus entre les candidats et le vecteur requête.** (6 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>pairwise_distances(candidates, vec, metric='cosine')</code></p>
</details>

In [ ]:
def query(vec, model, k, max_search_radius):
  
    data = model['data']
    table = model['table']
    random_vectors = model['random_vectors']
    num_vector = random_vectors.shape[1]
    
    # Calcul de l'index du bin du vecteur requête
    bin_index_bits = (vec.dot(random_vectors) >= 0).flatten()
   
    # Recherche progressive dans les bins voisins
    candidate_set = set()
    for search_radius in range(0, max_search_radius+1):
        candidate_set = search_nearby_bins(bin_index_bits, table, search_radius, initial_candidates=candidate_set)
   
    # Tri des candidats par distance
    nearest_neighbors = pd.DataFrame(candidate_set, columns=['id'])
    candidates = data[list(candidate_set),:]
    
    # --------------------------------------------------------------------------
    # ICI: Calculer les distances cosinus et les stocker dans nearest_neighbors['distance']
    # --------------------------------------------------------------------------
    nearest_neighbors['distance'] = None  # Remplacer None par le calcul correct
        
    return nearest_neighbors.nsmallest(k, 'distance'), len(candidate_set)

**Q7.2 - Exécuter une requête pour trouver les 10 plus proches voisins de Barack Obama avec un rayon de recherche de 3. Afficher le nom et la distance pour chaque voisin.** (6 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>query(corpus[id_obama,:], model, k=10, max_search_radius=3)</code> puis joindre avec le DataFrame wiki pour afficher les noms</p>
</details>

In [ ]:
# Compléter le code ~ 3-4 lignes de code

---
**Fin de l'examen #2**